In [11]:
# !pip install xgboost lightgbm catboost

In [58]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, confusion_matrix, classification_report, make_scorer, accuracy_score
from math import sqrt
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

In [13]:
clsf_data = pd.read_csv('../data/processed_smoke_detector.csv')
X_clsf = clsf_data.drop(['Fire Alarm'], axis=1)
y_clsf = clsf_data['Fire Alarm']
X_clsf_train, X_clsf_test, y_clsf_train, y_clsf_test = train_test_split(X_clsf, y_clsf, test_size=0.2)

In [14]:
X_clsf_train.columns = X_clsf_train.columns.str.replace(r'[\[\]<>]', '', regex=True)
X_clsf_test.columns = X_clsf_test.columns.str.replace(r'[\[\]<>]', '', regex=True)

In [15]:
regr_data = pd.read_csv('../data/processed_trip_duration.csv')
X_regr = regr_data.drop(['trip_duration'], axis=1)
y_regr = regr_data['trip_duration']
X_regr_train, X_regr_test, y_regr_train, y_regr_test = train_test_split(X_regr, y_regr, test_size=0.2)

# XGBoost

In [ ]:
model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1)

In [17]:
model.fit(X_clsf_train, y_clsf_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [18]:
y_clsf_pred = model.predict(X_clsf_test)

In [19]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1809    0]
 [   0 6441]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1809
           1       1.00      1.00      1.00      6441

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



In [53]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=200, learning_rate=0.5)

In [54]:
model.fit(X_regr_train, y_regr_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.5, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [55]:
y_regr_pred = model.predict(X_regr_test)

In [56]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 169.03822326660156
MSE: 58642.609375
RMSE: 242.1623615985771
MAPE: 0.528927212951938
R^2: 0.75


# LightGBM

In [71]:
train_data = lgb.Dataset(X_clsf_train, label=y_clsf_train)
params = {'boosting_type': 'gbdt', 'objective': 'binary', 'num_leaves': 30}

In [72]:
model = lgb.train(params, train_data, num_boost_round=100)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 25756, number of negative: 7241
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000467 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1839
[LightGBM] [Info] Number of data points in the train set: 32997, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.780556 -> initscore=1.268908
[LightGBM] [Info] Start training from score 1.268908


In [73]:
y_pred = model.predict(X_clsf_test).round()

In [74]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1809    0]
 [   0 6441]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1809
           1       1.00      1.00      1.00      6441

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



In [75]:
train_data = lgb.Dataset(X_regr_train, label=y_regr_train)
params = {'objective': 'regression', 'metric': 'mse'}

In [76]:
model = lgb.train(params, train_data, num_boost_round=200)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010242 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1336
[LightGBM] [Info] Number of data points in the train set: 524605, number of used features: 33
[LightGBM] [Info] Start training from score 748.530285


In [77]:
y_pred = model.predict(X_regr_test)

In [78]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 169.03822326660156
MSE: 58642.609375
RMSE: 242.1623615985771
MAPE: 0.528927212951938
R^2: 0.75


# CatBoost

In [86]:
model = CatBoostClassifier(iterations=100, learning_rate=0.1)

In [87]:
model.fit(X_clsf_train, y_clsf_train, eval_set=(X_clsf_test, y_clsf_test), verbose=10)

0:	learn: 0.6004979	test: 0.6005688	best: 0.6005688 (0)	total: 11.3ms	remaining: 1.11s
10:	learn: 0.1830881	test: 0.1825716	best: 0.1825716 (10)	total: 59.8ms	remaining: 484ms
20:	learn: 0.0666179	test: 0.0660546	best: 0.0660546 (20)	total: 91.1ms	remaining: 343ms
30:	learn: 0.0264380	test: 0.0258782	best: 0.0258782 (30)	total: 126ms	remaining: 280ms
40:	learn: 0.0117369	test: 0.0112934	best: 0.0112934 (40)	total: 158ms	remaining: 227ms
50:	learn: 0.0058144	test: 0.0054605	best: 0.0054605 (50)	total: 192ms	remaining: 185ms
60:	learn: 0.0033126	test: 0.0030207	best: 0.0030207 (60)	total: 227ms	remaining: 145ms
70:	learn: 0.0021329	test: 0.0018990	best: 0.0018990 (70)	total: 257ms	remaining: 105ms
80:	learn: 0.0015269	test: 0.0013461	best: 0.0013461 (80)	total: 289ms	remaining: 67.7ms
90:	learn: 0.0011497	test: 0.0010021	best: 0.0010021 (90)	total: 319ms	remaining: 31.6ms
99:	learn: 0.0009587	test: 0.0008365	best: 0.0008365 (99)	total: 344ms	remaining: 0us

bestTest = 0.0008365072199
bes

In [88]:
y_pred = model.predict(X_clsf_test)

In [89]:
print(confusion_matrix(y_clsf_test, y_clsf_pred))
print(classification_report(y_clsf_test, y_clsf_pred))

[[1809    0]
 [   0 6441]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1809
           1       1.00      1.00      1.00      6441

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250



In [90]:
model = CatBoostRegressor(iterations=200)

In [91]:
model.fit(X_regr_train, y_regr_train)

Learning rate set to 0.407545
0:	learn: 386.9636290	total: 26.3ms	remaining: 5.24s
1:	learn: 342.7194196	total: 48.8ms	remaining: 4.83s
2:	learn: 322.5339926	total: 71.2ms	remaining: 4.68s
3:	learn: 312.9942645	total: 93.7ms	remaining: 4.59s
4:	learn: 307.6543399	total: 118ms	remaining: 4.61s
5:	learn: 303.7286107	total: 140ms	remaining: 4.53s
6:	learn: 299.6360934	total: 164ms	remaining: 4.52s
7:	learn: 296.5895293	total: 188ms	remaining: 4.5s
8:	learn: 294.3554114	total: 207ms	remaining: 4.4s
9:	learn: 291.9017025	total: 231ms	remaining: 4.39s
10:	learn: 290.1218081	total: 254ms	remaining: 4.37s
11:	learn: 288.8863507	total: 274ms	remaining: 4.29s
12:	learn: 286.6862785	total: 295ms	remaining: 4.25s
13:	learn: 285.2253071	total: 320ms	remaining: 4.25s
14:	learn: 284.1761166	total: 339ms	remaining: 4.18s
15:	learn: 282.9843939	total: 362ms	remaining: 4.16s
16:	learn: 282.2062450	total: 380ms	remaining: 4.09s
17:	learn: 281.2797495	total: 403ms	remaining: 4.08s
18:	learn: 280.5204301	t

In [92]:
y_pred = model.predict(X_regr_test)

In [93]:
print(f'MAE: {mean_absolute_error(y_regr_test, y_regr_pred)}')
print(f'MSE: {mean_squared_error(y_regr_test, y_regr_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_regr_test, y_regr_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_regr_test, y_regr_pred))}')
print(f'R^2: {round(r2_score(y_regr_test, y_regr_pred),2)}')

MAE: 169.03822326660156
MSE: 58642.609375
RMSE: 242.1623615985771
MAPE: 0.528927212951938
R^2: 0.75


In [ ]:
# from IPython.display import display, Javascript
# display(Javascript('IPython.notebook.save_checkpoint();'))


# import time
# time.sleep(1)

# !git add .
# !git commit -m "feat: XGBoost, LightGBM, CatBoost realisation"
# !git push